# Amnesic Inference Validation — does Probe Ablation Improve Accuracy?

**Hypothesis**: Our factorial experiment (§3.5 of the paper) showed that ablating the source-letter probe direction at L17 T=10 *increased* source-kept by +15pp. This was measured with teacher-forced prefix. Does the effect generalize to **real inference on accuracy** (not just source preservation)?

If yes → first actual positive finding: **amnesic inference hack = free +X pp accuracy on MCQ reasoning**.

If no → confirms +15pp was artefact of teacher-forced protocol; paper stays as systematic null.

**Protocol** (~3 min runtime):
1. Load model + refit L17 probe (cached from Stage B)
2. Build 10 letter-direction vectors (PCA-back-projected logreg weights, unit norm)
3. Install hook on L17: at last prompt token, project out **all** letter directions (symmetric amnesic)
4. Eval: 100 SuperGPQA questions (≥2 correct rollouts, clean baseline), direct letter logit readout
5. Three configs:
   - **baseline** — no modification
   - **amnesic_all** — project out all 10 letter directions at last prompt token
   - **amnesic_argmax** — project out only the direction of the letter probe predicts

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/amnesic_inference')
OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
PATCH_LAYER = 17

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Refit L17 probe + build letter direction vectors

For each letter L, direction `d_L = pca.components.T @ logreg.coef[L] * scaler.scale_`, normalized to unit norm.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
print(f'{len(shards)} shards')

MIN_LEN = 200
POOL_WINDOW = 10
pooled, labels = [], []
for shard in tqdm(shards, desc='probe fit'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs = json.loads(meta['offsets'])[f'L{PATCH_LAYER}']
        rollouts = json.loads(meta['rollouts'])
        acts = f.get_tensor(f'acts_L{PATCH_LAYER}')
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            rollout_acts = acts[offs[r_idx]:offs[r_idx+1]].float().numpy()
            pooled.append(rollout_acts[:POOL_WINDOW].mean(axis=0))
            labels.append(r['pred'])

pooled = np.stack(pooled); labels = np.array(labels)
letters = sorted(set(labels))
print(f'Fit on {len(pooled)} rollouts, letters={letters}')

letter_to_int = {l: i for i, l in enumerate(letters)}
y = np.array([letter_to_int[l] for l in labels])
scaler = StandardScaler().fit(pooled)
pca = PCA(n_components=128, random_state=42).fit(scaler.transform(pooled))
logreg = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
    pca.transform(scaler.transform(pooled)), y)
print(f'Train score: {logreg.score(pca.transform(scaler.transform(pooled)), y):.3f}')

# Build letter direction vectors: unit-norm in residual space
directions = {}
for letter, i in letter_to_int.items():
    coef = logreg.coef_[i]  # [128]
    d_scaled = pca.components_.T @ coef
    d_raw = d_scaled * scaler.scale_  # back to residual space
    d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
    directions[letter] = torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16)

# Stack into [10, d_model]
d_stack = torch.stack([directions[L] for L in letters])  # [10, d_model]
print(f'd_stack shape: {d_stack.shape}')

# Probe on GPU for quick argmax during inference
probe_scaler_mean = torch.from_numpy(scaler.mean_.astype(np.float32)).cuda().to(torch.bfloat16)
probe_scaler_scale = torch.from_numpy(scaler.scale_.astype(np.float32)).cuda().to(torch.bfloat16)
probe_pca_T = torch.from_numpy(pca.components_.astype(np.float32)).cuda().to(torch.bfloat16)  # [128, d]
probe_coef = torch.from_numpy(logreg.coef_.astype(np.float32)).cuda().to(torch.bfloat16)  # [10, 128]
probe_intercept = torch.from_numpy(logreg.intercept_.astype(np.float32)).cuda().to(torch.bfloat16)  # [10]

def probe_predict(x):
    """x: [d_model] residual. Returns letter argmax index."""
    xs = (x.to(torch.float32) - probe_scaler_mean.to(torch.float32)) / probe_scaler_scale.to(torch.float32).clamp(min=1e-6)
    xp = (probe_pca_T.to(torch.float32) @ xs)  # [128]
    logits = probe_coef.to(torch.float32) @ xp + probe_intercept.to(torch.float32)  # [10]
    return logits.argmax().item()

## Cell 4 — Amnesic hook installer

Hook on L17 forward-output: at last prompt token position, project out either all letter directions (symmetric amnesic) or the probe-argmax direction.

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class AmnesicHook:
    def __init__(self):
        self.mode = 'off'  # 'off' | 'all' | 'argmax'
        self.target_pos = None  # absolute token position for ablation (use -1 for last)
        self.applied = False

    def set(self, mode, target_pos):
        self.mode = mode
        self.target_pos = target_pos
        self.applied = False

    def off(self):
        self.mode = 'off'
        self.applied = False

    def make_hook(self):
        def hook(module, inp, out):
            if self.mode == 'off' or self.applied: return out
            hidden = out[0] if isinstance(out, tuple) else out
            # hidden: [B, T, d]
            T = hidden.shape[1]
            pos = self.target_pos if self.target_pos >= 0 else T + self.target_pos
            if pos < 0 or pos >= T: return out
            hidden = hidden.clone()
            x = hidden[:, pos, :]  # [B, d]
            if self.mode == 'all':
                # Project out all letter directions: x_new = x - Σ (x · d_L) d_L
                # d_stack: [10, d] (bf16, unit norm)
                coefs = x @ d_stack.T  # [B, 10]
                delta = coefs @ d_stack  # [B, d]
                hidden[:, pos, :] = x - delta
            elif self.mode == 'argmax':
                # Use probe to identify argmax letter, ablate only that direction
                for b in range(x.shape[0]):
                    argmax_idx = probe_predict(x[b].float())
                    d = d_stack[argmax_idx]  # [d]
                    c = (x[b] * d).sum()
                    hidden[b, pos, :] = x[b] - c * d
            self.applied = True
            if isinstance(out, tuple):
                return (hidden,) + out[1:]
            return hidden
        return hook

amnesic = AmnesicHook()
layer_L17 = get_layer_module(model, PATCH_LAYER)
h_amnesic = layer_L17.register_forward_hook(amnesic.make_hook())
print(f'✅ amnesic hook installed on L{PATCH_LAYER}')

## Cell 5 — Load eval set + prompt formatter

In [ ]:
# Use same ≥2-correct filter as mythos notebook for clean baseline
questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rollouts_ = json.loads(meta['rollouts'])
        n_correct = sum(1 for r in rollouts_ if r['correct'])
        if n_correct >= 2:
            questions.append(dict(
                question=meta['question'],
                options=json.loads(meta['options']),
                gold_letter=meta['gold_letter'],
                n_options=len(json.loads(meta['options'])),
                n_correct=n_correct))
print(f'{len(questions)} eval questions')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

# Letter token IDs (single-token variants)
letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}
print('letter → token_id:', letter_ids)

## Cell 6 — Eval: baseline vs amnesic_all vs amnesic_argmax

In [ ]:
N_EVAL = 100
CONFIGS = ['baseline', 'amnesic_all', 'amnesic_argmax']

random.seed(42)
sample = random.sample(questions, min(N_EVAL, len(questions)))
print(f'Evaluating {len(sample)} questions × {len(CONFIGS)} configs')

results = []
t0 = time.time()
for i, q in enumerate(tqdm(sample, desc='amnesic eval')):
    try:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        last_pos = ids.shape[1] - 1

        row = dict(idx=i, gold=q['gold_letter'], n_options=q['n_options'])
        for config in CONFIGS:
            if config == 'baseline':
                amnesic.off()
            else:
                mode = 'all' if config == 'amnesic_all' else 'argmax'
                amnesic.set(mode, last_pos)
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {L: logits[letter_ids[L]].item() for L in 'ABCDEFGHIJ'[:q['n_options']]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'{config}_pred'] = pred
            row[f'{config}_correct'] = (pred == q['gold_letter'])
            row[f'{config}_gold_logit'] = letter_logits[q['gold_letter']]
        amnesic.off()
        results.append(row)
        if (i+1) % 10 == 0:
            accs = {c: sum(r[f'{c}_correct'] for r in results) / len(results) for c in CONFIGS}
            acc_str = ' | '.join(f'{c}={accs[c]*100:.0f}%' for c in CONFIGS)
            print(f'  [{i+1}/{N_EVAL}] {acc_str}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

with open(OUT/'amnesic_results.json', 'w') as f:
    json.dump(dict(n=len(results), configs=CONFIGS, results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ done in {(time.time()-t0)/60:.1f} min')

## Cell 7 — Analysis + verdict

In [ ]:
from scipy.stats import binomtest

print(f'=== Amnesic Inference on SuperGPQA (N={len(results)}) ===\n')
accs = {}
for config in CONFIGS:
    n_correct = sum(r[f'{config}_correct'] for r in results)
    acc = n_correct / len(results)
    accs[config] = acc
    print(f'{config:20s}: {n_correct}/{len(results)} = {acc*100:.1f}%')

baseline_acc = accs['baseline']
print(f'\n=== Δ vs baseline ===')
for config in CONFIGS:
    if config == 'baseline': continue
    delta = (accs[config] - baseline_acc) * 100
    # McNemar-style: count disagreements
    n_bp_only = sum(1 for r in results if r['baseline_correct'] and not r[f'{config}_correct'])
    n_c_only = sum(1 for r in results if not r['baseline_correct'] and r[f'{config}_correct'])
    # Binomial test on disagreements
    n_disagree = n_bp_only + n_c_only
    if n_disagree > 0:
        p_value = binomtest(n_c_only, n_disagree, p=0.5, alternative='two-sided').pvalue
    else:
        p_value = 1.0
    print(f'  {config}: Δ={delta:+.1f}pp | agreement_flips: {n_c_only} gained vs {n_bp_only} lost (p={p_value:.3f})')

# Verdict
print('\n=== VERDICT ===')
max_gain = max(accs[c] - baseline_acc for c in CONFIGS if c != 'baseline')
best_config = max([c for c in CONFIGS if c != 'baseline'], key=lambda c: accs[c])
gain_pp = max_gain * 100

if gain_pp > 5:
    print(f'✅ POSITIVE: {best_config} gives +{gain_pp:.1f}pp over baseline')
    print(f'   FIRST POSITIVE FINDING — amnesic inference as free accuracy boost')
    print(f'   Validates §3.5 Finding B on accuracy (not just source-kept)')
elif -2 < gain_pp < 5:
    print(f'⚖️  NEUTRAL: max gain {gain_pp:+.1f}pp — within noise')
    print(f'   Confirms +15pp source-kept was teacher-forced artifact, not accuracy gain')
    print(f'   Paper stays as systematic null + methodological contributions (LIP, amnesic probe)')
else:
    print(f'❌ NEGATIVE: {gain_pp:+.1f}pp — amnesic ablation HURTS accuracy')
    print(f'   Probe direction is load-bearing for accuracy (positive-correlational)')
    print(f'   Amnesic only reduces noise in source-kept, not actual accuracy')

# Also compute confidence stats
print(f'\n=== Mean gold logit ===')
for config in CONFIGS:
    mean_gold = np.mean([r[f'{config}_gold_logit'] for r in results])
    print(f'  {config}: {mean_gold:.3f}')